<h1 align=center style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
GTA VI
</font>
</h1>


<h2 align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
The Dataset
</font>
</h2>


<center>
<div dir="ltr" style="line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size="3">

| **Column Name** | **Description** |
| --- | --- |
| `transaction_id` | Unique identifier for each record in the dataset |
| `year` | Calendar year of the record |
| `month` | Calendar month of the record (1 to 12) |
| `quarter` | Financial quarter derived from the month (1 to 4) |
| `country` | Name of the country where the sale was made |
| `iso3_code` | Three-letter ISO code (ISO 3166-1 alpha-3) for the country |
| `region` | Continent or geographical region of the country |
| `platform` | Gaming platform (e.g., PS3, PS4, PC, Xbox Series X\|S) |
| `game_edition` | Purchased edition of the game (Standard, Premium, Legacy, etc.) |
| `units_sold` | Number of copies sold in that month/country/platform |
| `new_customers` | Number of buyers who purchased for the first time in that period |
| `returning_customers` | Number of returning buyers (those who have purchased before) |
| `estimated_active_players` | Estimated number of unique active players |
| `peak_concurrent_players` | Highest number of concurrent players recorded in that month |
| `online_players` | Number of active players in any of the online sections |
| `story_mode_players` | Number of players engaged with the story mode content (single-player) |
| `gta_online_players` | Number of active players specifically in the GTA Online section |
| `average_playtime_hours` | Average playtime hours per active player in that month |
| `average_session_length_minutes` | Average length of each gaming session in minutes |
| `holiday_season` | Was the sale during the holiday season (November, December, January)? (0 or 1) |
| `major_sale_event` | Name of the special sale event active in that month (if any) |
| `marketing_campaign` | Was a marketing campaign active during that period? (0 or 1) |
| `customer_rating` | Average rating submitted by users (from 4.2 to 5.0) |
| `review_count` | Number of submitted reviews |
| `refund_rate_percentage` | Percentage of refunded purchases |
| `currency` | Local currency code of the target country |
| `exchange_rate_to_usd` | Local currency to USD exchange rate at the time of sale |
| `internet_penetration_percentage` | Internet penetration percentage in the target country |
| `gaming_market_size` | Estimated relative size of the gaming market in the country |
| `population_millions` | Population of the target country in millions |
| `gdp_per_capita_usd` | GDP per capita of the country in USD (indicating purchasing power) |
| `release_phase` | Platform release life cycle phase (Launch, Growth, Mature, Legacy) |
| `weekend_sales_percentage` | Percentage share of sales on weekends |
| `weekday_sales_percentage` | Percentage share of sales on weekdays |
| `season` | Meteorological season (Winter, Spring, Summer, Autumn) |
| `special_event` | Special event that stimulated sales (e.g., release of a major update) |
| `top_game_category` | Main genre category of the game (Action-Adventure) |
| `platform_generation` | Gaming console generation (e.g., Gen7, Gen8, Gen9, PC) |
| `gross_revenue_usd` | **(First Target Column - Regression):** Gross revenue earned in USD |
| `sales_channel` | **(Second Target Column - Classification):** Sales channel, with values `Physical` or `Digital` |

</font>
</div>
</center>

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

train_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')

train_df, val_df = train_test_split(train_data, test_size=0.2, random_state=42)

<h2 align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
Feature Engineering
</font>
</h2>


In [5]:
from sklearn.preprocessing import OrdinalEncoder

train_df_final = train_df.copy()
val_df_final = val_df.copy()
test_data_final = test_data.copy()

for df in [train_df_final, val_df_final, test_data_final]:
    if 'units_sold' in df.columns:
        df['units_sold'] = pd.to_numeric(
            df['units_sold'].astype(str).str.replace(r'[\$,]', '', regex=True), 
            errors='coerce'
        )
    if 'gross_revenue_usd' in df.columns:
        df['gross_revenue_usd'] = pd.to_numeric(
            df['gross_revenue_usd'].astype(str).str.replace(r'[\$,]', '', regex=True), 
            errors='coerce'
        )

cat_cols = [
    'country', 'iso3_code', 'region', 'platform', 'game_edition',
    'currency', 'release_phase', 'season', 'platform_generation',
    'major_sale_event', 'special_event', 'top_game_category'
]

num_cols_to_fill = [
    'units_sold', 'new_customers', 'returning_customers',
    'estimated_active_players', 'gta_online_players', 'story_mode_players',
    'refund_rate_percentage', 'gdp_per_capita_usd', 'exchange_rate_to_usd'
]

def engineer_features(df):
    data = df.copy()
    
    for col in num_cols_to_fill:
        if col in data.columns:
            data[col] = pd.to_numeric(data[col], errors='coerce')
            data[col] = data[col].fillna(data[col].median())
    
    total_cust = data['new_customers'] + data['returning_customers']
    data['new_customer_ratio'] = data['new_customers'] / (total_cust + 1.0)
    data['gta_online_ratio'] = data['gta_online_players'] / (data['estimated_active_players'] + 1.0)
    data['story_ratio'] = data['story_mode_players'] / (data['estimated_active_players'] + 1.0)
    
    data['effective_units'] = data['units_sold'] * (1 - data['refund_rate_percentage'] / 100)
    data['market_power'] = (data['gdp_per_capita_usd'] * data['exchange_rate_to_usd']) / 1000
    
    min_year = 2013
    data['time_step'] = (data['year'] - min_year) * 12 + data['month']
    data['internet_x_time'] = data['internet_penetration_percentage'] * (data['year'] - min_year)
    data['month_sin'] = np.sin(2 * np.pi * data['month'] / 12.0)
    data['month_cos'] = np.cos(2 * np.pi * data['month'] / 12.0)
    
    data['has_sale_event'] = data['major_sale_event'].notna().astype(int)
    data['has_special_event'] = data['special_event'].notna().astype(int)
    
    data['gen_time_interact'] = data['time_step'] * (data['platform_generation'] == 'Gen9').astype(int)
    
    return data

train_feat = engineer_features(train_df_final)
val_feat = engineer_features(val_df_final)
test_feat = engineer_features(test_data_final)

for c in cat_cols:
    train_feat[c] = train_feat[c].fillna('Missing').astype(str)
    val_feat[c] = val_feat[c].fillna('Missing').astype(str)
    test_feat[c] = test_feat[c].fillna('Missing').astype(str)

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
train_feat[cat_cols] = encoder.fit_transform(train_feat[cat_cols])
val_feat[cat_cols] = encoder.transform(val_feat[cat_cols])
test_feat[cat_cols] = encoder.transform(test_feat[cat_cols])

drop_cols = ['transaction_id', 'sales_channel', 'gross_revenue_usd']
feature_names = [c for c in train_feat.columns if c not in drop_cols]

X_train_clf = train_feat[feature_names]
y_train_clf = (train_feat['sales_channel'] == 'Digital').astype(int)

X_val_clf = val_feat[feature_names]
y_val_clf = (val_feat['sales_channel'] == 'Digital').astype(int)

X_test = test_feat[feature_names]

mask_train_rev = train_feat['gross_revenue_usd'].notna()
mask_val_rev = val_feat['gross_revenue_usd'].notna()

X_train_reg = train_feat.loc[mask_train_rev, feature_names]
y_train_reg_ppu = (
    train_feat.loc[mask_train_rev, 'gross_revenue_usd'].astype(float) / 
    train_feat.loc[mask_train_rev, 'units_sold'].astype(float)
)

X_val_reg = val_feat.loc[mask_val_rev, feature_names]
y_val_reg_revenue = val_feat.loc[mask_val_rev, 'gross_revenue_usd'].astype(float)

<h2 align="left" style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
Modeling
</font>
</h2>

<p dir="ltr" style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size="3">
    Now that you have cleaned the data and possibly added or removed features from it, it is time to train a model that can predict the target variable of this problem.
</font>
</p>

In [8]:
from lightgbm import LGBMClassifier, LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostClassifier, CatBoostRegressor
import numpy as np

lgb_clf = LGBMClassifier(
    n_estimators=1200,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)
lgb_clf.fit(X_train_clf, y_train_clf)

cb_clf = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.04,
    depth=6,
    random_seed=42,
    verbose=0
)
cb_clf.fit(X_train_clf, y_train_clf)

train_clf_prob = 0.5 * lgb_clf.predict_proba(X_train_clf)[:, 1] + 0.5 * cb_clf.predict_proba(X_train_clf)[:, 1]

X_tr_reg_ext = X_train_reg.copy()
X_tr_reg_ext['prob_digital'] = train_clf_prob[mask_train_rev]

y_train_log_rev = np.log1p(train_feat.loc[mask_train_rev, 'gross_revenue_usd'].astype(float))

lgb_reg = LGBMRegressor(
    n_estimators=1500,
    learning_rate=0.02,
    num_leaves=31,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)
lgb_reg.fit(X_tr_reg_ext, y_train_log_rev)

xgb_reg = XGBRegressor(
    n_estimators=1500,
    learning_rate=0.02,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    tree_method='hist'
)
xgb_reg.fit(X_tr_reg_ext, y_train_log_rev)

cb_reg = CatBoostRegressor(
    iterations=1200,
    learning_rate=0.03,
    depth=6,
    random_seed=42,
    verbose=0
)
cb_reg.fit(X_tr_reg_ext, y_train_log_rev)

CatBoostRegressor(depth=6, iterations=1200, learning_rate=0.03, loss_function='RMSE', random_seed=42, verbose=0)

<h3 align="left" style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
Evaluation Metric
</font>
</h3>

<p dir="ltr" style="direction: ltr; text-align: justify; line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size="3">
    To evaluate the performance of your model before submitting the answer, you can use the <code>log_loss</code> (for the sales channel classification section) and <code>mean_squared_log_error</code> (for calculating RMSLE in the revenue prediction section) metrics from the <code>scikit-learn</code> library. 
    <br><br>
    It is recommended to simulate your score using the judging system's formula.
</font>
</p>

In [10]:
from sklearn.metrics import log_loss, mean_squared_log_error

val_prob_lgb = lgb_clf.predict_proba(X_val_clf)[:, 1]
val_prob_cb = cb_clf.predict_proba(X_val_clf)[:, 1]
val_prob_ensemble = 0.5 * val_prob_lgb + 0.5 * val_prob_cb

X_v_reg_ext = X_val_reg.copy()
X_v_reg_ext['prob_digital'] = val_prob_ensemble[mask_val_rev]

val_pred_log_lgb = lgb_reg.predict(X_v_reg_ext)
val_pred_log_xgb = xgb_reg.predict(X_v_reg_ext)
val_pred_log_cb = cb_reg.predict(X_v_reg_ext)

val_pred_log_ensemble = (
    0.34 * val_pred_log_lgb +
    0.33 * val_pred_log_xgb +
    0.33 * val_pred_log_cb
)
val_pred_revenue = np.expm1(val_pred_log_ensemble)

val_prob_clipped = np.clip(val_prob_ensemble, 1e-15, 1 - 1e-15)
loss_clf = log_loss(y_val_clf, val_prob_clipped)

val_pred_revenue_clipped = np.clip(val_pred_revenue, 0, None)
loss_reg = np.sqrt(mean_squared_log_error(y_val_reg_revenue, val_pred_revenue_clipped))

combined_error = (0.35 * loss_clf) + (0.65 * loss_reg)
competition_score = 100 * np.exp(-1.3 * combined_error)

print(f"Log Loss (Classification) : {loss_clf:.4f}")
print(f"RMSLE (Regression)        : {loss_reg:.4f}")
print(f"Combined Competition Score : {competition_score:.2f} / 100")

Log Loss (Classification) : 0.1344
RMSLE (Regression)        : 0.0599
Combined Competition Score : 89.42 / 100


<h2 align="left" style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
 Prediction for Test Data and Output
</font>
</h2>

<p dir="ltr" style="direction: ltr;text-align: left;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size="3">
    After feature engineering and modeling, you have an algorithm that can take you from independent variables to target variables.
    <br>
    Use this model to predict the samples present in the test data and prepare the results in the format of the table (<code>dataframe</code>) below.
</font>
</p>

<div dir="ltr" style="direction: ltr;text-align: left;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size="3">
    
| Column | Description |
|:------:|:---:|
|transaction_id|Unique transaction identifier (exactly the same as the test data)|
|gross_revenue_usd|Your prediction for the revenue amount in USD (decimal number)|
|sales_channel|Your prediction for the sales channel (Physical or Digital)|
    
</font>
</div>

<p dir="ltr" style="direction: ltr;text-align: left;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size="3">
    The dataframe name must be <i>submission</i>; otherwise, the judging system will not be able to evaluate your attempt.
    <br>
    This dataframe includes 3 columns named <i>transaction_id</i>, <i>gross_revenue_usd</i>, and <i>sales_channel</i> and has exactly 3696 rows.
    <br>
    For every row present in the <i>test</i> dataframe, you must have predicted the revenue and sales channel values.
    <br>
    The table below shows the first 5 rows of the <code>submission</code> dataframe. Of course, in your answer, the predicted values may be different.
</font>
</p>

<div style="text-align: left;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size="3">
    
||transaction_id|gross_revenue_usd|sales_channel|
|----|-----|-----|-----|
|0|GTA5-110039|12500.50|Physical|
|1|GTA5-110005|8400.00|Digital|
|2|GTA5-110093|2100.75|Physical|
|3|GTA5-110072|15600.00|Digital|
|4|GTA5-110034|450.25|Physical|

</font>
</div>

In [12]:
test_prob_lgb = lgb_clf.predict_proba(X_test)[:, 1]
test_prob_cb = cb_clf.predict_proba(X_test)[:, 1]
test_prob_ensemble = 0.5 * test_prob_lgb + 0.5 * test_prob_cb

pred_sales_channel = np.where(test_prob_ensemble >= 0.5, 'Digital', 'Physical')

X_test_reg_ext = X_test.copy()
X_test_reg_ext['prob_digital'] = test_prob_ensemble

pred_test_log = (
    0.34 * lgb_reg.predict(X_test_reg_ext) +
    0.33 * xgb_reg.predict(X_test_reg_ext) +
    0.33 * cb_reg.predict(X_test_reg_ext)
)
pred_gross_revenue = np.clip(np.expm1(pred_test_log), 0, None)

submission = pd.DataFrame({
    'transaction_id': test_data_final['transaction_id'],
    'gross_revenue_usd': pred_gross_revenue,
    'sales_channel': pred_sales_channel
})

submission.head()

,transaction_id,gross_revenue_usd,sales_channel
0,GTA5-112598,4117.980897,Digital
1,GTA5-104880,12203.623687,Digital
2,GTA5-113648,7585.588277,Digital
3,GTA5-111846,13273.292855,Digital
4,GTA5-109886,2095.233045,Digital


<h2 align="left" style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
<b>Result Generator Cell</b>
</font>
</h2>

<p dir="ltr" style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size="3">
    Run the cell below to generate the <code>result.zip</code> file.
    <br>
    You must upload the <code>result.zip</code> file in Quera and wait for the judging result!
    <br>
    This file includes <code>submission.csv</code>, this very notebook you are coding in, and your saved model.
</font>
</p>

In [13]:
import zipfile

def compress(file_names):
    print("File Paths:")
    print(file_names)
    compression = zipfile.ZIP_DEFLATED
    with zipfile.ZipFile("result.zip", mode="w") as zf:
        for file_name in file_names:
            zf.write('./' + file_name, file_name, compress_type=compression)


submission.to_csv('submission.csv', index=False)

file_names = ['submission.csv', 'GTA.ipynb']
compress(file_names)

File Paths:
['submission.csv', 'GTA.ipynb']
